# Scaffold-based 5-Fold Cross-Validation

Uses the normalised data:
- SOURCE: normalized_data/{prop}_source.csv (Log10 + Z-score done)
- TARGET: normalized_data/{prop}_target.csv (Log10 only) → Z-score with per-fold train statistics


In [1]:
import pandas as pd
import os
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold

RDLogger.DisableLog("rdApp.*")

source_dir = r'../data/normalized_data'
output_dir = r'../data/cv_datasets'
os.makedirs(output_dir, exist_ok=True)

PROPERTIES = ['clearance', 'half_life', 'fu']
N_FOLDS = 5

print(f"Source: {source_dir}")
print(f"Output: {output_dir}\n")

Source: ..\data\normalized_data
Output: ..\data\cv_datasets



## Scaffold split function

In [2]:
def get_scaffold(smi):
    try:
        m = Chem.MolFromSmiles(str(smi))
        if m is None:
            return None
        return MurckoScaffold.MurckoScaffoldSmiles(mol=m)
    except:
        return None

def scaffold_k_fold_split(df, k=5, seed=0):
    """Scaffold-based k-fold split"""
    np.random.seed(seed)
    
    df = df.copy()
    df['scaffold'] = df['smiles'].map(get_scaffold)
    df = df.dropna(subset=['scaffold'])
    
    # Group by scaffold
    scaffolds = df['scaffold'].unique()
    scaffold_indices = {s: df[df['scaffold'] == s].index.tolist() for s in scaffolds}
    
    # Assign to folds largest first (balancing)
    scaffold_sizes = [(s, len(idx)) for s, idx in scaffold_indices.items()]
    scaffold_sizes.sort(key=lambda x: -x[1])
    
    fold_indices = [[] for _ in range(k)]
    for scaffold, _ in scaffold_sizes:
        smallest_fold = min(range(k), key=lambda i: len(fold_indices[i]))
        fold_indices[smallest_fold].extend(scaffold_indices[scaffold])
    
    # Train/test split
    splits = []
    for test_fold_idx in range(k):
        test_indices = fold_indices[test_fold_idx]
        train_indices = []
        for i in range(k):
            if i != test_fold_idx:
                train_indices.extend(fold_indices[i])
        
        train_df = df.iloc[train_indices].drop('scaffold', axis=1).reset_index(drop=True)
        test_df = df.iloc[test_indices].drop('scaffold', axis=1).reset_index(drop=True)
        splits.append((train_df, test_df))
    
    return splits

## Build the CV splits

In [3]:
summary_data = []
norm_stats = []

for prop in PROPERTIES:
    print(f"\n{'='*100}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*100}")
    
    # Load SOURCE
    src_path = os.path.join(source_dir, f"{prop}_source.csv")
    if not os.path.exists(src_path):
        print(f"✗ Source file not found: {src_path}")
        continue
    
    source_df = pd.read_csv(src_path)
    print(f"Source: {len(source_df)} rows")
    
    # Load TARGET
    tgt_path = os.path.join(source_dir, f"{prop}_target.csv")
    if not os.path.exists(tgt_path):
        print(f"✗ Target file not found: {tgt_path}")
        continue
    
    target_df = pd.read_csv(tgt_path)
    print(f"Target: {len(target_df)} rows")
    
    # K-Fold Split
    print(f"\nScaffold-based {N_FOLDS}-Fold Split:")
    splits = scaffold_k_fold_split(target_df, k=N_FOLDS, seed=42)
    
    # Save each fold
    for fold_idx, (train_df, test_df) in enumerate(splits):
        fold_dir = os.path.join(output_dir, f"fold_{fold_idx}")
        os.makedirs(fold_dir, exist_ok=True)
        
        # Save SOURCE (identical for every fold)
        src_save = os.path.join(fold_dir, f"{prop}_source.csv")
        source_df.to_csv(src_save, index=False)
        
        # TARGET Z-score: transform both train and test with this fold's train statistics (prevents test leakage)
        mu = train_df['value'].mean()
        sd = train_df['value'].std()
        train_df = train_df.assign(value=(train_df['value'] - mu) / sd)
        test_df = test_df.assign(value=(test_df['value'] - mu) / sd)
        norm_stats.append({'Fold': fold_idx, 'Property': prop, 'log10_mean': mu, 'log10_std': sd})

        # Save TARGET train/test
        train_save = os.path.join(fold_dir, f"{prop}_target_train.csv")
        test_save = os.path.join(fold_dir, f"{prop}_target_test.csv")
        train_df.to_csv(train_save, index=False)
        test_df.to_csv(test_save, index=False)
        
        print(f"  Fold {fold_idx}: train {len(train_df):5d} | test {len(test_df):4d}")
        
        summary_data.append({
            'Fold': fold_idx,
            'Property': prop,
            'Source': len(source_df),
            'Target_Train': len(train_df),
            'Target_Test': len(test_df),
            'Target_Total': len(train_df) + len(test_df)
        })


Property: CLEARANCE
Source: 14387 rows
Target: 1027 rows

Scaffold-based 5-Fold Split:
  Fold 0: train   821 | test  206
  Fold 1: train   821 | test  206
  Fold 2: train   822 | test  205
  Fold 3: train   822 | test  205
  Fold 4: train   822 | test  205

Property: HALF_LIFE
Source: 9204 rows
Target: 1164 rows

Scaffold-based 5-Fold Split:
  Fold 0: train   931 | test  233
  Fold 1: train   931 | test  233
  Fold 2: train   931 | test  233
  Fold 3: train   931 | test  233
  Fold 4: train   932 | test  232

Property: FU
Source: 3783 rows
Target: 353 rows

Scaffold-based 5-Fold Split:
  Fold 0: train   282 | test   71
  Fold 1: train   282 | test   71
  Fold 2: train   282 | test   71
  Fold 3: train   283 | test   70
  Fold 4: train   283 | test   70


## Summary

In [4]:
summary_df = pd.DataFrame(summary_data)

print("\n" + "="*120)
print("5-FOLD SCAFFOLD CV SUMMARY")
print("="*120)
print(summary_df.to_string(index=False))

print(f"\n{'='*120}")
print(f"Output: {output_dir}")
print(f"{'='*120}")

summary_df.to_csv(os.path.join(output_dir, 'cv_summary.csv'), index=False)
# For inverting predictions: log10(value) = z * log10_std + log10_mean
pd.DataFrame(norm_stats).to_csv(os.path.join(output_dir, 'target_norm_stats.csv'), index=False)
print(f"\n✓ CV split complete")
print(f"  - 5 folds created")
print(f"  - 3 properties (clearance, half_life, fu)")
print(f"  - Each fold: source (full) + target_train + target_test")


5-FOLD SCAFFOLD CV SUMMARY
 Fold  Property  Source  Target_Train  Target_Test  Target_Total
    0 clearance   14387           821          206          1027
    1 clearance   14387           821          206          1027
    2 clearance   14387           822          205          1027
    3 clearance   14387           822          205          1027
    4 clearance   14387           822          205          1027
    0 half_life    9204           931          233          1164
    1 half_life    9204           931          233          1164
    2 half_life    9204           931          233          1164
    3 half_life    9204           931          233          1164
    4 half_life    9204           932          232          1164
    0        fu    3783           282           71           353
    1        fu    3783           282           71           353
    2        fu    3783           282           71           353
    3        fu    3783           283           70           3